In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 01 \u2014 Gaussian Geometry"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Objectives\n",
        "- Connect Gaussian parameters to visible shapes.\n",
        "- Compare shifts, scales, and covariance matrices.\n",
        "- Use eigendecomposition to draw covariance ellipses."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Mathematical Background\n",
        "$p(x)=\\frac{1}{\\sqrt{2\\pi\\sigma^2}}\\exp[-(x-\\mu)^2/(2\\sigma^2)]$ and $\\mathcal N(x\\mid\\mu,\\Sigma)$. Integrals remain one because parameters redistribute density rather than create probability mass."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Implementation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import sys\n",
        "from pathlib import Path\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "PROJECT = Path.cwd()\n",
        "if PROJECT.name == 'notebooks':\n",
        "    PROJECT = PROJECT.parent\n",
        "FIGURES = PROJECT / 'figures'\n",
        "FIGURES.mkdir(exist_ok=True)\n",
        "if str(PROJECT) not in sys.path:\n",
        "    sys.path.insert(0, str(PROJECT))\n",
        "\n",
        "plt.style.use('seaborn-v0_8-whitegrid')\n",
        "COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']\n",
        "RNG = np.random.default_rng(7)\n",
        "\n",
        "def savefig(name):\n",
        "    plt.tight_layout()\n",
        "    plt.savefig(FIGURES / name, dpi=180, bbox_inches='tight', transparent=True)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Experiments"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from src.gaussian import sample_gaussian\n",
        "from src.utils import covariance_ellipse\n",
        "\n",
        "x = np.linspace(-6, 6, 600)\n",
        "mu, sigma = 0.0, 1.0\n",
        "samples = RNG.normal(mu, sigma, size=4000)\n",
        "pdf = np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))\n",
        "plt.figure(figsize=(7,4))\n",
        "plt.hist(samples, bins=45, density=True, alpha=.35, color=COLORS[0], label='sample histogram')\n",
        "plt.plot(x, pdf, color=COLORS[3], lw=2.5, label='theoretical density')\n",
        "plt.title('Does the histogram approach the Gaussian density?')\n",
        "plt.xlabel('x'); plt.ylabel('density'); plt.legend(); savefig('gaussian_histogram.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "plt.figure(figsize=(7,4))\n",
        "for i, m in enumerate([-2,0,2]):\n",
        "    plt.plot(x, np.exp(-0.5*((x-m)/1)**2)/np.sqrt(2*np.pi), lw=2.2, color=COLORS[i], label=f'mu={m}')\n",
        "plt.title('Changing \u03bc translates density without changing area')\n",
        "plt.xlabel('x'); plt.ylabel('density'); plt.legend(); savefig('mu_variation.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "plt.figure(figsize=(7,4))\n",
        "for i, s in enumerate([.5,1,2]):\n",
        "    plt.plot(x, np.exp(-0.5*(x/s)**2)/(s*np.sqrt(2*np.pi)), lw=2.2, color=COLORS[i], label=f'sigma={s}')\n",
        "plt.title('Changing \u03c3 trades peak height for spread; area stays one')\n",
        "plt.xlabel('x'); plt.ylabel('density'); plt.legend(); savefig('sigma_variation.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "covariances = [np.array([[1,0],[0,1]]), np.array([[4,0],[0,1]]), np.array([[1,.8],[.8,1]])]\n",
        "labels = ['identity: round', 'anisotropic: stretched', 'correlated: rotated']\n",
        "fig, axes = plt.subplots(1,3,figsize=(12,4),sharex=True,sharey=True)\n",
        "for ax, cov, label, color in zip(axes,covariances,labels,COLORS):\n",
        "    X = sample_gaussian([0,0], cov, 900, random_state=RNG)\n",
        "    ax.scatter(X[:,0], X[:,1], s=8, alpha=.35, color=color)\n",
        "    ax.plot(*covariance_ellipse(cov).T, color='black', lw=2)\n",
        "    vals, vecs = np.linalg.eigh(cov)\n",
        "    ax.set_title(label + f'\\n\u03bb={np.round(vals,2)}')\n",
        "    ax.set_aspect('equal')\n",
        "savefig('covariance_shapes.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "cov = np.array([[1,.8],[.8,1]])\n",
        "vals, vecs = np.linalg.eigh(cov)\n",
        "ellipse = covariance_ellipse(cov, n_std=2)\n",
        "plt.figure(figsize=(5,5))\n",
        "plt.plot(ellipse[:,0], ellipse[:,1], color=COLORS[3], lw=2.5)\n",
        "for val, vec in zip(vals, vecs.T):\n",
        "    end = 2*np.sqrt(val)*vec\n",
        "    plt.arrow(0,0,end[0],end[1],head_width=.08,length_includes_head=True,color='black')\n",
        "plt.title('Which directions explain covariance ellipse axes?')\n",
        "plt.axis('equal'); plt.xlabel('x1'); plt.ylabel('x2'); savefig('covariance_ellipse.png')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Observations\n",
        "- Mean shifts location; variance controls spread; covariance rotates and stretches density through eigenvectors and eigenvalues.\n",
        "- Normalization keeps total probability equal to one even as the shape changes."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Key Takeaways\n",
        "- A Gaussian is a geometric object as much as a formula.\n",
        "- One ellipse cannot represent separated or multimodal structure."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Transition to the next notebook\n",
        "Next we combine several Gaussians and interpret data generation with latent component assignments."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "pygments_lexer": "ipython3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}